# 01 — Data Analysis

**Goal:** load the portfolio, compute log returns, and characterise their statistical properties before any modelling begins.

**Portfolio:** `SPY` (equities) · `TLT` (rates) · `GLD` (gold) · `USO` (oil)

**Steps:**
1. Download / load cached prices
2. Compute log returns
3. Plot cumulative returns
4. Examine the return distribution (histogram, skew, kurtosis)
5. Plot rolling volatility (visualise volatility clustering)
6. Plot the correlation matrix across assets
7. Examine drawdowns


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import download_prices, compute_log_returns, portfolio_returns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')


## 1. Load data

Set `use_cache=False` on the first run to download fresh data; switch to `True` on later runs to avoid re-hitting the API.

In [ ]:
TICKERS = ['SPY', 'TLT', 'GLD', 'USO']
START_DATE = '2015-01-01'

prices = download_prices(TICKERS, start=START_DATE, cache_path='../data/market_prices.csv')
prices.tail()


In [ ]:
returns = compute_log_returns(prices)
returns.describe()


## 2. Cumulative returns

Plot the growth of $1 invested in each asset since the start date — a quick visual for how differently these four asset classes have behaved.

In [ ]:
cumulative = (1 + returns).cumprod()

fig, ax = plt.subplots(figsize=(10, 5))
cumulative.plot(ax=ax)
ax.set_title('Cumulative Returns by Asset')
ax.set_ylabel('Growth of $1')
plt.savefig('../results/figures/cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Return distribution

Check skewness and (excess) kurtosis — financial returns are almost never normally distributed, and this matters directly for choosing between normal and Student-t Parametric VaR later in `03_var_models.ipynb`.

In [ ]:
from scipy.stats import skew, kurtosis

stats_table = pd.DataFrame({
    'mean': returns.mean(),
    'std': returns.std(),
    'skew': returns.apply(skew),
    'excess_kurtosis': returns.apply(kurtosis),  # kurtosis() already subtracts 3
})
stats_table


In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(16, 3.5))
for ax, ticker in zip(axes, TICKERS):
    sns.histplot(returns[ticker], bins=60, kde=True, ax=ax)
    ax.set_title(ticker)
plt.tight_layout()
plt.savefig('../results/figures/return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Rolling volatility — visualising volatility clustering

The key stylised fact that motivates GARCH-family models: large moves tend to be followed by large moves (of either sign), and calm periods tend to persist too.

In [ ]:
rolling_vol = returns.rolling(21).std() * np.sqrt(252)  # annualised, 21-day rolling window

fig, ax = plt.subplots(figsize=(10, 5))
rolling_vol.plot(ax=ax)
ax.set_title('21-Day Rolling Annualised Volatility')
ax.set_ylabel('Annualised Volatility')
plt.savefig('../results/figures/rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Correlation matrix

Correlations across asset classes drive diversification — and, crucially, tend to shift (often converging toward 1) exactly during the crisis periods you'll be stress-testing in `05_stress_testing.ipynb`. Worth revisiting this same chart on a stress-period subset later.

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Return Correlation Matrix')
plt.savefig('../results/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Drawdowns

The peak-to-trough decline over time — a more intuitive risk measure for a non-technical reader than volatility alone, and a good chart for the risk report's executive summary.

In [ ]:
def compute_drawdown(cum_returns: pd.Series) -> pd.Series:
    running_max = cum_returns.cummax()
    return (cum_returns - running_max) / running_max

drawdowns = cumulative.apply(compute_drawdown)

fig, ax = plt.subplots(figsize=(10, 5))
drawdowns.plot(ax=ax)
ax.set_title('Drawdowns by Asset')
ax.set_ylabel('Drawdown')
plt.savefig('../results/figures/drawdowns.png', dpi=150, bbox_inches='tight')
plt.show()

print('Worst drawdown per asset:')
drawdowns.min()


## 7. Equal-weighted portfolio returns

Combine the four assets into a single portfolio return series, used throughout the rest of the project. Revisit the weights later — an equal-weighted 25/25/25/25 split is a reasonable starting point, not a considered allocation.

In [ ]:
weights = {'SPY': 0.25, 'TLT': 0.25, 'GLD': 0.25, 'USO': 0.25}
port_returns = portfolio_returns(returns, weights)

port_returns.to_csv('../data/portfolio_returns.csv')
port_returns.describe()


---

**Next notebook:** `02_volatility_models.ipynb` — fit Historical, EWMA, GARCH(1,1) and EGARCH volatility models to the portfolio return series and compare their out-of-sample forecast accuracy.